# Testing Pandas for EBD Processing

Note: this data is pulling from an unzipped txt EBD file in the "ebd" folder not included in the repository

## Resources

- [Pandas vs R Cheatsheet](https://datascientyst.com/pandas-vs-r-cheat-sheet/)
- [Combining/Merge/Axis with dataframes](https://www.geeksforgeeks.org/pandas/how-to-combine-two-dataframe-in-python-pandas/)

In [199]:
import pandas as pd
import json
import numpy as np
import geopandas as gpd


# eBird Dataset file name
ebd_fn = 'ebd_US-NC_202101_202602_unv_smp_relMar-2026.txt'

# define data types
dtypes = {'LAST EDITED DATE' : str, 'COUNTRY' : str, 'COUNTRY CODE' : str,
'STATE' : str, 'STATE CODE' : str, 'COUNTY' : str, 'COUNTY CODE' : str,
'IBA CODE' : str, 'BCR CODE' : str, 'USFWS CODE' : str, 'ATLAS BLOCK' : str,
'LOCALITY' : str, 'LOCALITY ID' : str, 'LOCALITY TYPE' : str,
'LATITUDE' : float, 'LONGITUDE' : float, 'OBSERVATION DATE' : str,
'TIME OBSERVATIONS STARTED' : str, 'OBSERVER ID' : str, 
'OBSERVER ORCID ID' : str, 'SAMPLING EVENT IDENTIFIER' : str,
'OBSERVATION TYPE' : str, 'PROTOCOL NAME' : str, 'PROTOCOL CODE' : str,
'PROJECT NAMES' : object, 'PROJECT IDENTIFIERS' : object, 'DURATION MINUTES' : float,
'EFFORT DISTANCE KM' : str, 'EFFORT AREA HA' : str, 'NUMBER OBSERVERS' : float,
'ALL SPECIES REPORTED' : str, 'GROUP IDENTIFIER' : str, 'HAS MEDIA' : str,
'CHECKLIST COMMENTS' : str, 'GLOBAL UNIQUE IDENTIFIER' : str, 
'TAXONOMIC ORDER' : str, 'CATEGORY' : str, 'TAXON CONCEPT ID' : str,
'COMMON NAME' : str, 'SCIENTIFIC NAME' : str, 'SUBSPECIES COMMON NAME' : str,
'SUBSPECIES SCIENTIFIC NAME' : str, 'EXOTIC CODE' : str, 
'OBSERVATION COUNT' : str, 'BREEDING CODE' : str, 'BREEDING CATEGORY' : str,
'BEHAVIOR CODE' : str, 'AGE/SEX' : str, 'APPROVED' : str, 'REVIEWED' : str,
'REASON' : str, 'SPECIES COMMENTS' : str}

## Load the EBD into a Pandas DF

In [200]:


# ebd_raw = pd.read_csv(f'./ebd/{ebd_fn}', sep='\t', dtype = dtypes)

## Create Test Dataset

In [201]:
# create subset for testing
## PRODUCTION
# ebd = ebd_raw
## END PRODUCTION

### TEST
ebd = pd.read_csv("ebd_test.csv", dtype = dtypes)

## save data
# ebd = pd.concat([ebd_raw.head(1000), ebd_raw.tail(1000)])
# ebd.to_csv("ebd_test.csv")
# ebd_bigger = pd.concat([ebd_raw.head(10000), ebd_raw.tail(10000)])
# ebd_bigger.to_csv("ebd_test_bigger.csv")

### END TEST

ebd = ebd.fillna('')




## Start Exploring

In [202]:
rows_columns = ebd.shape
print(f'{rows_columns[0]:,d} rows, {rows_columns[1]} columns found')
# ebd['SAMPLING EVENT IDENTIFIER'].sort_values()

# remove spaces from colum names
ebd.columns = ebd.columns.str.replace(' ', '_')

# get column names
ebd_columns =  ebd.columns

# define observation fields
observation_fields = pd.Series([
  "GLOBAL_UNIQUE_IDENTIFIER",
  "TAXONOMIC_ORDER",
  "CATEGORY",
  "TAXON_CONCEPT_ID",
  "COMMON_NAME",
  "SCIENTIFIC_NAME",
  "SUBSPECIES_COMMON_NAME",
  "SUBSPECIES_SCIENTIFIC_NAME",
  "EXOTIC_CODE",
  "OBSERVATION_COUNT",
  "BREEDING_CODE",
  "BREEDING_CATEGORY",
  "BEHAVIOR_CODE",
  "AGE/SEX",
  "APPROVED",
  "REVIEWED",
  "REASON",
  "SPECIES_COMMENTS"
])

# get checklist fields
checklist_fields = pd.Series(ebd_columns[~ebd_columns.isin(observation_fields)])

# add SAMPLING EVENT IDENTIFIER to observations, remove Unamed column from checklist_fields
# observation_fields = pd.concat([observation_fields, pd.Series(['SAMPLING_EVENT_IDENTIFIER'])])
# checklist_fields = checklist_fields[checklist_fields != 'Unnamed: 52']

print(f'checklist fields:\n{list(checklist_fields)}')
print(f'observation fields:\n{list(observation_fields)}')

2,000 rows, 54 columns found
checklist fields:
['Unnamed:_0', 'LAST_EDITED_DATE', 'COUNTRY', 'COUNTRY_CODE', 'STATE', 'STATE_CODE', 'COUNTY', 'COUNTY_CODE', 'IBA_CODE', 'BCR_CODE', 'USFWS_CODE', 'ATLAS_BLOCK', 'LOCALITY', 'LOCALITY_ID', 'LOCALITY_TYPE', 'LATITUDE', 'LONGITUDE', 'OBSERVATION_DATE', 'TIME_OBSERVATIONS_STARTED', 'OBSERVER_ID', 'OBSERVER_ORCID_ID', 'SAMPLING_EVENT_IDENTIFIER', 'OBSERVATION_TYPE', 'PROTOCOL_NAME', 'PROTOCOL_CODE', 'PROJECT_NAMES', 'PROJECT_IDENTIFIERS', 'DURATION_MINUTES', 'EFFORT_DISTANCE_KM', 'EFFORT_AREA_HA', 'NUMBER_OBSERVERS', 'ALL_SPECIES_REPORTED', 'GROUP_IDENTIFIER', 'HAS_MEDIA', 'CHECKLIST_COMMENTS', 'Unnamed:_52']
observation fields:
['GLOBAL_UNIQUE_IDENTIFIER', 'TAXONOMIC_ORDER', 'CATEGORY', 'TAXON_CONCEPT_ID', 'COMMON_NAME', 'SCIENTIFIC_NAME', 'SUBSPECIES_COMMON_NAME', 'SUBSPECIES_SCIENTIFIC_NAME', 'EXOTIC_CODE', 'OBSERVATION_COUNT', 'BREEDING_CODE', 'BREEDING_CATEGORY', 'BEHAVIOR_CODE', 'AGE/SEX', 'APPROVED', 'REVIEWED', 'REASON', 'SPECIES_COMM

## Augment Checklist Data

### Get Block Info

In [203]:
# get block layer

gpd_blocks = gpd.read_file("blocks.geojson")
gpd_blocks.shape

(5453, 41)

In [204]:
# subset ebd for checklists, lat/lon only
ebd_checklists = ebd.groupby([
    'SAMPLING_EVENT_IDENTIFIER',
    'LATITUDE',
    'LONGITUDE'
    ]).size().reset_index()

gpd_checklists = gpd.GeoDataFrame(
    ebd_checklists,
    geometry = gpd.points_from_xy(
        ebd_checklists.LONGITUDE,
        ebd_checklists.LATITUDE
    ),
    crs = "EPSG:4326"
)

# spatial join with blocks to get block info

checklist_blocks = gpd_checklists.sjoin(
    gpd_blocks, how = 'left')

# checklist_blocks.head()
# print(list(checklist_blocks.columns))

checklist_blocks['PRIORITY_BLOCK'] = np.where(
    checklist_blocks.block_type == 'block_type', "1", "0"
)
ncba_fields = checklist_blocks[['SAMPLING_EVENT_IDENTIFIER', 'ID_NCBA_BLOCK', 'ID_BLOCK_CODE', 'PRIORITY_BLOCK']]
ncba_fields = ncba_fields.fillna('').reset_index(drop = True)



### Generate NCBA Fields

** THIS NEEDS WORK **

In [205]:
# # # add fields
# # ncba_fields['YEAR'] = ncba_fields['OBSERVATION_DATE'].str[:4].astype(int)
# # ncba_fields['MONTH'] = ncba_fields['OBSERVATION_DATE'].str[5:7].astype(int)
# # ncba_fields['PROJECT_CODE'] = np.where(
# #     ncba_fields['PROJECT_NAMES'].str.contains('North Carolina Bird Atlas'),
# #     'EBIRD_ATL_NC',
# #     'EBIRD'
# # )
# # # ncba_fields['ID_NCBA_BLOCK'] = np.where(
# # #     ncba_fields[ncba_fields]
# # # )
# # # ncba_fields['ID_BLOCK_CODE'] = ""
# # # ncba_fields['PRIORITY_BLOCK'] = ""
# # ncba_fields['NCBA_JULIAN_DAY'] = ""
# # ncba_fields['NCBA_WEEK'] = ""
# # ncba_fields['NCBA_SEASON'] = ""
# # ncba_fields['NCBA_QUARTER'] = ""
# # ncba_fields['NCBA_DATE_ADDED'] = "2026-04-27"
# # ncba_fields['NCBA_EBD_VER'] = "Mar-2026"
# # ncba_fields['NCBA_OBSERVER'] = ""

# # ncba_fields.head()


## Tinker with Grouping by Checklist

In [206]:
# # obs_only = ebd[observation_fields]
# # check_only = ebd[checklist_fields]


# # File for test.json
# out_json = open('test.json', 'w', encoding = 'utf-8-sig')

# # add fields
# ebd['YEAR'] = ebd['OBSERVATION_DATE'].str[:4].astype(int)
# ebd['MONTH'] = ebd['OBSERVATION_DATE'].str[5:7].astype(int)
# ebd['PROJECT_CODE'] = np.where(
#     ebd['PROJECT_NAMES'].str.contains('North Carolina Bird Atlas'),
#     'EBIRD_ATL_NC',
#     'EBIRD'
# )
# # ebd['ID_NCBA_BLOCK'] = np.where(
# #     ncba_fields[ncba_fields]
# # )
# # ebd['ID_BLOCK_CODE'] = ""
# # ebd['PRIORITY_BLOCK'] = ""
# ebd['NCBA_JULIAN_DAY'] = ""
# ebd['NCBA_WEEK'] = ""
# ebd['NCBA_SEASON'] = ""
# ebd['NCBA_QUARTER'] = ""
# ebd['NCBA_DATE_ADDED'] = "2026-04-27"
# ebd['NCBA_EBD_VER'] = "Mar-2026"
# ebd['NCBA_OBSERVER'] = ""


# ebd_grouped = (
#     # ebd.groupby(checklist_fields)
#     ebd.groupby([
#         'SAMPLING_EVENT_IDENTIFIER', 'LAST_EDITED_DATE', 'COUNTRY',
#         'COUNTRY_CODE', 'STATE', 'STATE_CODE', 'COUNTY', 'COUNTY_CODE',
#         'IBA_CODE', 'BCR_CODE', 'USFWS_CODE', 'ATLAS_BLOCK', 'LOCALITY',
#         'LOCALITY_ID', 'LOCALITY_TYPE', 'LATITUDE', 'LONGITUDE',
#         'OBSERVATION_DATE', 'TIME_OBSERVATIONS_STARTED', 'OBSERVER_ID',
#         'PROTOCOL_NAME', 'PROTOCOL_CODE', 'PROJECT_CODE', 
#         'PROJECT_NAMES', 
#         'DURATION_MINUTES', 'EFFORT_DISTANCE_KM', 'EFFORT_AREA_HA',
#         'NUMBER_OBSERVERS', 'ALL_SPECIES_REPORTED', 'GROUP_IDENTIFIER',
#         'CHECKLIST_COMMENTS', 'YEAR', 'MONTH', 'NCBA_JULIAN_DAY', 'NCBA_WEEK',
#         'NCBA_SEASON',
#         'NCBA_DATE_ADDED', 'NCBA_EBD_VER', 'NCBA_OBSERVER'
#         ])
#     # ebd.groupby(['SAMPLING EVENT IDENTIFIER'])
#     .apply(lambda x: x[list(observation_fields)].to_dict('records'))
#     .reset_index()
#     .rename(columns={0:'OBSERVATIONS'})
#     # .to_dict(orient='records')
# )

# # ebd['ID_NCBA_BLOCK'] = np.where(
# #     ncba_fields[ncba_fields]
# # )
# ebd_grouped.head()
# # test = ebd_grouped['SAMPLING_EVENT_IDENTIFIER'].groupby('SAMPLING_EVENT_IDENTIFIER').size()
# # print(test)

# # ebd_grouped = pd.merge(
# #     ebd_grouped,
# #     ncba_fields,
# #     how='left',
# #     on = 'SAMPLING_EVENT_IDENTFIER'
# # )

# # ebd_json = ebd_grouped.to_dict(orient='records')
# # out_json.write(json.dumps(ebd_json, indent = 2))

# # ebd_grouped.shape
# # ebd_grouped.head()


## Group Observation Fields

Add SEI and group by this value.

In [207]:
# make a list of obs fields plus SEI
sei = pd.Series(['SAMPLING_EVENT_IDENTIFIER'])
observation_fields = pd.concat([observation_fields, sei])
obs_only = ebd[observation_fields]

obs_grouped = (
    obs_only.groupby('SAMPLING_EVENT_IDENTIFIER')
    .apply(lambda x: x[list(observation_fields)].to_dict('records'))
    .reset_index()
    .rename(columns={0:'OBSERVATIONS'})
)

C:\Users\skanderson\AppData\Local\Temp\1\ipykernel_10988\873890228.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[list(observation_fields)].to_dict('records'))


## Group Checklist Fields

Summarize each field.

In [208]:
# # add fields
# ebd['YEAR'] = ebd['OBSERVATION_DATE'].str[:4].astype(int)
# ebd['MONTH'] = ebd['OBSERVATION_DATE'].str[5:7].astype(int)
# ebd['PROJECT_CODE'] = np.where(
#     ebd['PROJECT_NAMES'].str.contains('North Carolina Bird Atlas'),
#     'EBIRD_ATL_NC',
#     'EBIRD'
# )
# # ebd['ID_NCBA_BLOCK'] = np.where(
# #     ncba_fields[ncba_fields]
# # )
# # ebd['ID_BLOCK_CODE'] = ""
# # ebd['PRIORITY_BLOCK'] = ""
# ebd['NCBA_JULIAN_DAY'] = ""
# ebd['NCBA_WEEK'] = ""
# ebd['NCBA_SEASON'] = ""
# ebd['NCBA_QUARTER'] = ""
# ebd['NCBA_DATE_ADDED'] = "2026-04-27"
# ebd['NCBA_EBD_VER'] = "Mar-2026"
# ebd['NCBA_OBSERVER'] = ""

In [209]:
# get checklists only, collapse fields that are repeats
check_only = (
    ebd[checklist_fields] # get only checklist fields
    .groupby('SAMPLING_EVENT_IDENTIFIER')
    .first()
)

# add block and other NCBA fields
check_only = pd.merge(
    check_only,
    ncba_fields,
    how = 'left',
    on = 'SAMPLING_EVENT_IDENTIFIER'
)
check_only.shape
print(check_only.head())

  SAMPLING_EVENT_IDENTIFIER  Unnamed:_0            LAST_EDITED_DATE  \
0                S297587800    20051488  2026-02-01 17:36:01.035857   
1                S297593603    20051571  2026-02-05 13:30:42.603676   
2                S297594414    20052145  2026-02-01 18:08:53.448845   
3                S297643082    20051588  2026-02-02 02:05:21.176707   
4                S297692628    20051788  2026-02-02 09:27:23.174982   

         COUNTRY COUNTRY_CODE           STATE STATE_CODE  COUNTY COUNTY_CODE  \
0  United States           US  North Carolina      US-NC  Yancey   US-NC-199   
1  United States           US  North Carolina      US-NC  Yancey   US-NC-199   
2  United States           US  North Carolina      US-NC  Yancey   US-NC-199   
3  United States           US  North Carolina      US-NC  Yancey   US-NC-199   
4  United States           US  North Carolina      US-NC  Yancey   US-NC-199   

    IBA_CODE  ... EFFORT_AREA_HA NUMBER_OBSERVERS ALL_SPECIES_REPORTED  \
0             ... 

## Join Observations and Checklists

In [ ]:
ebd_out = pd.merge(
    check_only,
    ncba_fields,
    how = 'outer',
    on = 'SAMPLING_EVENT_IDENTIFIER'
)

ebd_out = pd.merge(
    ebd_out,
    obs_only,
    how = 'outer',
    on = 'SAMPLING_EVENT_IDENTIFIER'
)

## SOMEHOW THIS RESULTS IN MULTIPLE SEI rows...

print(ebd_out.info())
print(ebd_out.head())


ebd_json = ebd_out.to_dict(orient='records')
# File for test.json
out_json = open('test.json', 'w', encoding = 'utf-8-sig')
out_json.write(json.dumps(ebd_json, indent = 2))

ebd_out.to_csv("ebd_out.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 60 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   SAMPLING_EVENT_IDENTIFIER   2000 non-null   object 
 1   Unnamed:_0                  2000 non-null   int64  
 2   LAST_EDITED_DATE            2000 non-null   object 
 3   COUNTRY                     2000 non-null   object 
 4   COUNTRY_CODE                2000 non-null   object 
 5   STATE                       2000 non-null   object 
 6   STATE_CODE                  2000 non-null   object 
 7   COUNTY                      2000 non-null   object 
 8   COUNTY_CODE                 2000 non-null   object 
 9   IBA_CODE                    2000 non-null   object 
 10  BCR_CODE                    2000 non-null   object 
 11  USFWS_CODE                  2000 non-null   object 
 12  ATLAS_BLOCK                 2000 non-null   object 
 13  LOCALITY                    2000 